In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy


name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-9327fcef-38d8-daf7-d486-e79cefc0275a, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [2]:
GITHUB_USERNAME = "comepotyyy"

assert (
    GITHUB_USERNAME != "YOUR_GITHUB_USERNAME"
    and len(GITHUB_USERNAME.strip()) > 0
), "Set your real GitHub username."

print(f"Configured student GitHub identity: {GITHUB_USERNAME}")


Configured student GitHub identity: comepotyyy


In [3]:
import subprocess
import hashlib
from mpi4py import MPI
from numba import cuda

In [5]:

def init_distributed_context(github_user: str):
    """
    Initialize MPI Cartesian topology and bind each rank
    to GPU device 0.

    Returns the MPI context, neighbors, coordinates,
    and hardware-bound instance seed.
    """

    comm = MPI.COMM_WORLD

    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError(
            f"Expected exactly 4 MPI ranks, got {size}"
        )

    dims = [2, 2]
    periods = [False, False]

    cart_comm = comm.Create_cart(
        dims=dims,
        periods=periods,
        reorder=False
    )

    coords = cart_comm.Get_coords(rank)

    row_rank, col_rank = coords

    nbr_up, nbr_down = cart_comm.Shift(0, 1)

    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None

    gpu_uuid = cart_comm.bcast(gpu_uuid, root=0)

    instance_seed = derive_student_seed(
        github_user,
        gpu_uuid
    )

    print(
        f"[Rank {rank}] "
        f"Coordinates={coords}, "
        f"Up={nbr_up}, Down={nbr_down}, "
        f"Left={nbr_left}, Right={nbr_right}, "
        f"GPU=0"
    )

    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "rank": rank,
        "size": size,
        "coords": coords,
        "row_rank": row_rank,
        "col_rank": col_rank,
        "nbr_up": nbr_up,
        "nbr_down": nbr_down,
        "nbr_left": nbr_left,
        "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed
    }


In [6]:

%%writefile distributed_cuda_solver.py

import os
import sys
import json
import math
import hashlib
import subprocess

import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2

GLOBAL_NX = 512
GLOBAL_NY = 512

STEPS = 400
LAMBDA_DAMP = 0.15


# ============================================================
# HELPER 1: GET GPU UUID
# ============================================================

def get_gpu_uuid() -> str:
    """Get active NVIDIA GPU UUID."""

    out = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=uuid",
            "--format=csv,noheader"
        ],
        encoding="utf-8"
    )

    return out.strip().splitlines()[0].strip()


# ============================================================
# HELPER 2: GENERATE STUDENT SEED
# ============================================================

def derive_student_seed(
    github_user: str,
    gpu_uuid: str
) -> int:
    """Generate deterministic 64-bit student seed."""

    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()

    return int(digest[:16], 16)


# ============================================================
# TASK 1: DISTRIBUTED CARTESIAN TOPOLOGY
# ============================================================

def init_distributed_context(github_user: str):

    # 1. Initialize MPI
    comm = MPI.COMM_WORLD

    rank = comm.Get_rank()
    size = comm.Get_size()

    # 2. Verify exactly four ranks
    if size != 4:
        raise RuntimeError(
            f"Expected exactly 4 MPI ranks, got {size}"
        )

    # 3. Create 2x2 Cartesian process grid
    dims = [2, 2]
    periods = [False, False]

    cart_comm = comm.Create_cart(
        dims=dims,
        periods=periods,
        reorder=False
    )

    # 4. Get rank coordinates
    coords = cart_comm.Get_coords(rank)

    row_rank, col_rank = coords

    # 5. Find vertical neighbors
    nbr_up, nbr_down = cart_comm.Shift(0, 1)

    # 6. Find horizontal neighbors
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    # 7. Bind process to GPU 0
    cuda.select_device(0)

    # 8. Obtain GPU UUID on rank 0
    gpu_uuid = get_gpu_uuid() if rank == 0 else None

    # Broadcast UUID to other ranks
    gpu_uuid = cart_comm.bcast(gpu_uuid, root=0)

    # 9. Derive personal seed
    instance_seed = derive_student_seed(
        github_user,
        gpu_uuid
    )

    # 10. Display information
    print(
        f"[Rank {rank}] "
        f"Coordinates={coords}, "
        f"Up={nbr_up}, "
        f"Down={nbr_down}, "
        f"Left={nbr_left}, "
        f"Right={nbr_right}, "
        f"GPU=0, "
        f"Seed={instance_seed}",
        flush=True
    )

    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "rank": rank,
        "size": size,
        "coords": coords,
        "row_rank": row_rank,
        "col_rank": col_rank,
        "nbr_up": nbr_up,
        "nbr_down": nbr_down,
        "nbr_left": nbr_left,
        "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed
    }


# ============================================================
# TASK 1: INITIAL EXECUTION TEST
# ============================================================

if __name__ == "__main__":

    github_user = sys.argv[1]

    context = init_distributed_context(github_user)

    if context["rank"] == 0:
        print("\nTASK 1 INITIALIZATION COMPLETED", flush=True)
        print("MPI ranks:", context["size"], flush=True)
        print("Cartesian dimensions: 2 x 2", flush=True)
        print("GPU UUID:", context["gpu_uuid"], flush=True)
        print("INSTANCE_SEED:", context["instance_seed"], flush=True)


Writing distributed_cuda_solver.py


In [8]:

import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:")
    print(proc.stderr)

    raise RuntimeError(
        f"MPI execution failed: {proc.returncode}"
    )


STDOUT:

STDERR:



RuntimeError: MPI execution failed: 1

In [9]:

import os
import subprocess

print("1. Checking solver file...")
print("File exists:", os.path.exists("distributed_cuda_solver.py"))

print("\n2. Checking GitHub username...")
print("Username:", GITHUB_USERNAME)

print("\n3. Checking MPI...")
subprocess.run(["mpirun", "--version"])

print("\n4. Checking GPU...")
subprocess.run(["nvidia-smi"])

print("\n5. Running MPI with detailed error capture...")

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\n========== RETURN CODE ==========")
print(proc.returncode)

print("\n========== STDOUT ==========")
print(repr(proc.stdout))

print("\n========== STDERR ==========")
print(repr(proc.stderr))


1. Checking solver file...
File exists: True

2. Checking GitHub username...
Username: comepotyyy

3. Checking MPI...

4. Checking GPU...

5. Running MPI with detailed error capture...

========== RETURN CODE ==========
1

========== STDOUT ==========
''

========== STDERR ==========
''


In [10]:

import subprocess
import os

# Test 1: Check MPI executable
print("MPI executable:")
subprocess.run(["which", "mpirun"])

# Test 2: Check MPI version
result = subprocess.run(
    ["mpirun", "--version"],
    capture_output=True,
    text=True
)
print("\nMPI version output:")
print("Return code:", result.returncode)
print("STDOUT:", repr(result.stdout))
print("STDERR:", repr(result.stderr))

# Test 3: Launch four simple processes
print("\nTesting four MPI processes...")

test_code = """
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

print(f"Hello from rank {rank} of {size}", flush=True)
"""

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "-c", test_code
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=60
)

print("\nMPI TEST RESULT")
print("Return code:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)


MPI executable:

MPI version output:
Return code: 0
STDOUT: 'mpirun (Open MPI) 4.1.6\n\nReport bugs to http://www.open-mpi.org/community/help/\n'
STDERR: ''

Testing four MPI processes...

MPI TEST RESULT
Return code: 1
STDOUT:
 
STDERR:
 


In [11]:

import sys
import mpi4py
from mpi4py import MPI

print("Python executable:", sys.executable)
print("mpi4py version:", mpi4py.__version__)
print("\nMPI library used by Python:")
print(MPI.Get_library_version())


Python executable: /usr/bin/python3
mpi4py version: 4.1.2

MPI library used by Python:
Open MPI v4.1.6, package: Debian OpenMPI, ident: 4.1.6, repo rev: v4.1.6, Sep 30, 2023 


In [12]:

!which mpirun
!mpirun --allow-run-as-root --oversubscribe -np 4 python3 -c "from mpi4py import MPI; print('Hello from rank', MPI.COMM_WORLD.Get_rank(), 'of', MPI.COMM_WORLD.Get_size(), flush=True)"
!echo "Exit code: $?"


/usr/bin/mpirun
Exit code: 0


In [13]:
import os
import subprocess
import textwrap

# Create a tiny independent MPI test file
test_file = "/tmp/mpi_hello.py"

with open(test_file, "w") as f:
    f.write(textwrap.dedent("""
        from mpi4py import MPI
        import socket
        import sys

        comm = MPI.COMM_WORLD
        rank = comm.Get_rank()
        size = comm.Get_size()

        print(
            f"Hello from rank {rank}/{size} | "
            f"Host: {socket.gethostname()} | "
            f"Python: {sys.executable}",
            flush=True
        )

        comm.Barrier()

        if rank == 0:
            print("MPI TEST COMPLETED", flush=True)
    """))

print("Starting MPI test...")
print("Command: mpirun -np 4 python3", test_file)

result = subprocess.run(
    [
        "mpirun",
        "--allow-run-as-root",
        "--oversubscribe",
        "--tag-output",
        "-np", "4",
        "python3",
        test_file
    ],
    text=True,
    timeout=60
)

print("\nMPI return code:", result.returncode)

Starting MPI test...
Command: mpirun -np 4 python3 /tmp/mpi_hello.py

MPI return code: 1


In [14]:
import subprocess
import sys
import os

test_file = "/tmp/mpi_hello.py"

# Test 1: Run the Python script directly (without MPI)
print("===== TEST 1: Direct Python execution =====")

direct = subprocess.run(
    [sys.executable, test_file],
    capture_output=True,
    text=True,
    timeout=30
)

print("Return code:", direct.returncode)
print("STDOUT:", repr(direct.stdout))
print("STDERR:", repr(direct.stderr))


# Test 2: Run the same script with MPI and capture errors
print("\n===== TEST 2: MPI execution =====")

mpi = subprocess.run(
    [
        "mpirun",
        "--allow-run-as-root",
        "--oversubscribe",
        "--tag-output",
        "-np", "4",
        sys.executable,
        test_file
    ],
    capture_output=True,
    text=True,
    timeout=60
)

print("Return code:", mpi.returncode)
print("STDOUT:", repr(mpi.stdout))
print("STDERR:", repr(mpi.stderr))


# Test 3: Show MPI-related environment variables
print("\n===== TEST 3: MPI environment =====")

for key, value in sorted(os.environ.items()):
    if any(word in key.upper() for word in ["OMPI", "MPI", "PMIX", "PRTE"]):
        print(f"{key}={value}")

===== TEST 1: Direct Python execution =====
Return code: 0
STDOUT: 'Hello from rank 0/1 | Host: d95cf94efef4 | Python: /usr/bin/python3\nMPI TEST COMPLETED\n'
STDERR: ''

===== TEST 2: MPI execution =====
Return code: 1
STDOUT: ''
STDERR: ''

===== TEST 3: MPI environment =====


In [15]:
import subprocess
import sys

commands = [
    [
        "mpirun",
        "--allow-run-as-root",
        "--oversubscribe",
        "--mca", "plm", "isolated",
        "--mca", "btl", "self,tcp",
        "--mca", "pml", "ob1",
        "--tag-output",
        "-np", "4",
        sys.executable,
        "/tmp/mpi_hello.py"
    ],
    [
        "mpirun",
        "--allow-run-as-root",
        "--oversubscribe",
        "--mca", "orte_base_help_aggregate", "0",
        "--mca", "plm_base_verbose", "100",
        "-np", "4",
        sys.executable,
        "/tmp/mpi_hello.py"
    ]
]

for i, cmd in enumerate(commands, 1):
    print(f"\n========== TEST {i} ==========")
    print("Command:", " ".join(cmd))

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=60
        )

        print("Return code:", result.returncode)
        print("STDOUT:\n", result.stdout or "(empty)")
        print("STDERR:\n", result.stderr or "(empty)")

    except subprocess.TimeoutExpired:
        print("Timed out after 60 seconds")


========== TEST 1 ==========
Command: mpirun --allow-run-as-root --oversubscribe --mca plm isolated --mca btl self,tcp --mca pml ob1 --tag-output -np 4 /usr/bin/python3 /tmp/mpi_hello.py
Return code: 1
STDOUT:
 (empty)
STDERR:
 (empty)

========== TEST 2 ==========
Command: mpirun --allow-run-as-root --oversubscribe --mca orte_base_help_aggregate 0 --mca plm_base_verbose 100 -np 4 /usr/bin/python3 /tmp/mpi_hello.py
Return code: 1
STDOUT:
 (empty)
STDERR:
 (empty)


In [16]:
import subprocess

tests = {
    "Test A — hostname": [
        "mpirun", "--allow-run-as-root", "--oversubscribe",
        "-np", "2", "hostname"
    ],
    "Test B — echo": [
        "mpirun", "--allow-run-as-root", "--oversubscribe",
        "-np", "2", "/bin/echo", "MPI_PROCESS_STARTED"
    ],
    "Test C — MPI environment information": [
        "ompi_info", "--version"
    ],
    "Test D — launcher location": [
        "which", "orted"
    ],
}

for name, command in tests.items():
    print(f"\n===== {name} =====")

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=30
        )

        print("Exit code:", result.returncode)
        print("STDOUT:", repr(result.stdout))
        print("STDERR:", repr(result.stderr))

    except Exception as error:
        print(type(error).__name__, str(error))


===== Test A — hostname =====
Exit code: 1
STDOUT: ''
STDERR: ''

===== Test B — echo =====
Exit code: 1
STDOUT: ''
STDERR: ''

===== Test C — MPI environment information =====
Exit code: 0
STDOUT: 'Open MPI v4.1.6\n\nhttp://www.open-mpi.org/community/help/\n'
STDERR: ''

===== Test D — launcher location =====
Exit code: 0
STDOUT: '/usr/bin/orted\n'
STDERR: ''


In [17]:
import subprocess

commands = {
    "TEST 1 — Open MPI installation": [
        "ompi_info", "--version"
    ],
    "TEST 2 — Process launcher": [
        "orted", "--help"
    ],
    "TEST 3 — MPI runtime components": [
        "ompi_info", "--param", "plm", "all"
    ],
    "TEST 4 — MPI execution with hostname": [
        "mpirun",
        "--allow-run-as-root",
        "--oversubscribe",
        "-np", "2",
        "/bin/hostname"
    ],
    "TEST 5 — Open MPI shared libraries": [
        "ldd", "/usr/bin/orted"
    ]
}

for name, command in commands.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=30
        )

        print("Return code:", result.returncode)
        print("STDOUT:")
        print(result.stdout[:5000] or "(empty)")
        print("STDERR:")
        print(result.stderr[:5000] or "(empty)")

    except Exception as e:
        print("ERROR:", repr(e))


TEST 1 — Open MPI installation
Return code: 0
STDOUT:
Open MPI v4.1.6

http://www.open-mpi.org/community/help/

STDERR:
(empty)

TEST 2 — Process launcher
Return code: 1
STDOUT:
(empty)
STDERR:
Usage: orted [OPTION]...
Start an Open RTE Daemon

-d|--debug               Debug the OpenRTE
   --daemonize           Daemonize the orted into the background
   --debug-daemons       Enable debugging of OpenRTE daemons
   --debug-daemons-file  Enable debugging of OpenRTE daemons, storing output
                         in files
   -gmca|--gmca <arg0> <arg1>  
                         Pass global MCA parameters that are applicable to
                         all contexts (arg0 is the parameter name; arg1 is
                         the parameter value)
-h|--help                This help message
   --hnp                 Direct the orted to act as the HNP
   --hnp-uri <arg0>      URI for the HNP
   -mca|--mca <arg0> <arg1>  
                         Pass context-specific MCA parameters; they are


In [18]:
import subprocess

tests = {
    "TEST 1 — MPI launcher libraries":
        ["ldd", "/usr/bin/mpirun"],

    "TEST 2 — Runtime daemon libraries":
        ["ldd", "/usr/bin/orted"],

    "TEST 3 — hwloc library":
        ["ldd", "/usr/local/lib/libhwloc.so.15"],

    "TEST 4 — Open MPI installation paths":
        ["ompi_info", "--path", "all"],

    "TEST 5 — MPI launcher verbose diagnostics":
        [
            "mpirun",
            "--allow-run-as-root",
            "--oversubscribe",
            "--mca", "plm_base_verbose", "100",
            "--mca", "ess_base_verbose", "100",
            "--mca", "odls_base_verbose", "100",
            "-np", "2",
            "/bin/hostname"
        ]
}

for name, cmd in tests.items():
    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=40
        )

        print("Exit code:", result.returncode)
        print("STDOUT:\n", result.stdout[:7000] or "(empty)")
        print("STDERR:\n", result.stderr[:7000] or "(empty)")

    except Exception as e:
        print("Exception:", repr(e))


TEST 1 — MPI launcher libraries
Exit code: 0
STDOUT:
 	linux-vdso.so.1 (0x00007ffcf17fe000)
	libopen-rte.so.40 => /lib/x86_64-linux-gnu/libopen-rte.so.40 (0x00007dfb12022000)
	libopen-pal.so.40 => /lib/x86_64-linux-gnu/libopen-pal.so.40 (0x00007dfb11f6e000)
	libevent_core-2.1.so.7 => /lib/x86_64-linux-gnu/libevent_core-2.1.so.7 (0x00007dfb11f39000)
	libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x00007dfb11d26000)
	libz.so.1 => /lib/x86_64-linux-gnu/libz.so.1 (0x00007dfb11d0a000)
	libhwloc.so.15 => /usr/local/lib/libhwloc.so.15 (0x00007dfb11a00000)
	libevent_pthreads-2.1.so.7 => /lib/x86_64-linux-gnu/libevent_pthreads-2.1.so.7 (0x00007dfb11d03000)
	/lib64/ld-linux-x86-64.so.2 (0x00007dfb120f5000)
	libm.so.6 => /lib/x86_64-linux-gnu/libm.so.6 (0x00007dfb11917000)

STDERR:
 (empty)

TEST 2 — Runtime daemon libraries
Exit code: 0
STDOUT:
 	linux-vdso.so.1 (0x00007ffc109f2000)
	libopen-rte.so.40 => /lib/x86_64-linux-gnu/libopen-rte.so.40 (0x00007fec97672000)
	libc.so.6 => /lib/x86_64-linu

In [19]:
import subprocess
import os

tests = {
    "TEST 1 — ESS components":
        ["ompi_info", "--param", "ess", "all", "--level", "9"],

    "TEST 2 — Available Open MPI component files":
        ["bash", "-lc", "find /usr/lib/x86_64-linux-gnu/openmpi/lib/openmpi3 -maxdepth 1 -type f | sort"],

    "TEST 3 — Open MPI launcher libraries":
        ["ldd", "/usr/bin/mpirun"],

    "TEST 4 — Open MPI runtime environment":
        ["bash", "-lc", "env | sort | grep -Ei '^(LD_|OMPI_|OPAL_|ORTE_|PMIX_|PRTE_)' || true"],

    "TEST 5 — Open MPI ESS selection":
        [
            "mpirun",
            "--allow-run-as-root",
            "--oversubscribe",
            "--mca", "ess", "hnp",
            "--mca", "plm", "isolated",
            "--tag-output",
            "-np", "2",
            "/bin/hostname"
        ]
}

for name, cmd in tests.items():
    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=40
        )

        print("Exit code:", result.returncode)
        print("STDOUT:\n", result.stdout[:8000] or "(empty)")
        print("STDERR:\n", result.stderr[:8000] or "(empty)")

    except Exception as e:
        print("Exception:", repr(e))


TEST 1 — ESS components
Exit code: 0
STDOUT:
                  MCA ess: pmi (MCA v2.1.0, API v3.0.0, Component v4.1.6)
                 MCA ess: singleton (MCA v2.1.0, API v3.0.0, Component v4.1.6)
                 MCA ess: hnp (MCA v2.1.0, API v3.0.0, Component v4.1.6)
                 MCA ess: slurm (MCA v2.1.0, API v3.0.0, Component v4.1.6)
                 MCA ess: env (MCA v2.1.0, API v3.0.0, Component v4.1.6)
                 MCA ess: tool (MCA v2.1.0, API v3.0.0, Component v4.1.6)
            MCA ess base: ---------------------------------------------------
            MCA ess base: parameter "ess" (current value: "singleton", data source: environment, level: 2 user/detail, type: string)
                          Default selection set of components for the ess framework (<none> means use all components that can be found)
            MCA ess base: parameter "orte_ess_jobid" (current value: "", data source: default, level: 9 dev/all, type: string, synonym of: ess_base_jobid)
    

In [21]:
import os
import sys
import subprocess

clean_env = os.environ.copy()

for key in list(clean_env.keys()):
    if key.startswith(("OMPI_", "PMIX_", "ORTE_", "OPAL_", "PRTE_")):
        clean_env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "ess", "hnp",
    "--tag-output",
    "-np", "4",
    sys.executable,
    "distributed_cuda_solver.py",
    GITHUB_USERNAME
]

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=120,
    env=clean_env
)

print("Return code:", proc.returncode)
print("STDOUT:\n", proc.stdout or "(empty)")
print("STDERR:\n", proc.stderr or "(empty)")

if proc.returncode != 0:
    raise RuntimeError("MPI solver execution failed.")

Return code: 0
STDOUT:
 [1,0]<stdout>:[Rank 0] Coordinates=[0, 0], Up=-2, Down=2, Left=-2, Right=1, GPU=0, Seed=7843665258511339110
[1,2]<stdout>:[Rank 2] Coordinates=[1, 0], Up=0, Down=-2, Left=-2, Right=3, GPU=0, Seed=7843665258511339110
[1,0]<stdout>:
[1,0]<stdout>:TASK 1 INITIALIZATION COMPLETED
[1,0]<stdout>:MPI ranks: 4
[1,0]<stdout>:Cartesian dimensions: 2 x 2
[1,0]<stdout>:GPU UUID: GPU-9327fcef-38d8-daf7-d486-e79cefc0275a
[1,0]<stdout>:INSTANCE_SEED: 7843665258511339110
[1,1]<stdout>:[Rank 1] Coordinates=[0, 1], Up=-2, Down=3, Left=0, Right=-2, GPU=0, Seed=7843665258511339110
[1,3]<stdout>:[Rank 3] Coordinates=[1, 1], Up=1, Down=-2, Left=2, Right=-2, GPU=0, Seed=7843665258511339110

STDERR:
 (empty)


In [22]:
import os
import shutil

source = "distributed_cuda_solver.py"
backup = "distributed_cuda_solver_task1_backup.py"

assert os.path.exists(source), f"{source} not found!"
shutil.copy2(source, backup)

print("Backup created:", backup)
print("Current file size:", os.path.getsize(source), "bytes")

Backup created: distributed_cuda_solver_task1_backup.py
Current file size: 3873 bytes


In [23]:
with open("distributed_cuda_solver.py", "r") as f:
    print(f.read())


import os
import sys
import json
import math
import hashlib
import subprocess

import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2

GLOBAL_NX = 512
GLOBAL_NY = 512

STEPS = 400
LAMBDA_DAMP = 0.15


# ============================================================
# HELPER 1: GET GPU UUID
# ============================================================

def get_gpu_uuid() -> str:
    """Get active NVIDIA GPU UUID."""

    out = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=uuid",
            "--format=csv,noheader"
        ],
        encoding="utf-8"
    )

    return out.strip().splitlines()[0].strip()


# ============================================================
# HELPER 2: GENERATE STUDENT SEED
# ========================================

In [24]:
task2_code = r'''
# ============================================================
# TASK 2: SEEDED HETEROGENEOUS SUBDOMAIN INITIALIZATION
# ============================================================

LOCAL_NX = GLOBAL_NX // 2
LOCAL_NY = GLOBAL_NY // 2

DX = 1.0 / (GLOBAL_NX + 1)
DY = 1.0 / (GLOBAL_NY + 1)


def initialize_subdomains(context):
    """
    Create rank-local initial fields with one ghost-cell layer.

    Array indexing:
        axis 0 = y / rows
        axis 1 = x / columns

    Owned cells: [1:257, 1:257]
    Ghost cells: index 0 and index 257
    """

    row_rank = context["row_rank"]
    col_rank = context["col_rank"]
    seed = context["instance_seed"]

    nbr_up = context["nbr_up"]
    nbr_down = context["nbr_down"]
    nbr_left = context["nbr_left"]
    nbr_right = context["nbr_right"]

    # --------------------------------------------------------
    # 1. Global coordinates for local indices 0..257
    # --------------------------------------------------------

    local_rows = np.arange(LOCAL_NY + 2)
    local_cols = np.arange(LOCAL_NX + 2)

    global_rows = row_rank * LOCAL_NY + local_rows - 1
    global_cols = col_rank * LOCAL_NX + local_cols - 1

    y = global_rows * DY
    x = global_cols * DX

    X, Y = np.meshgrid(x, y)

    # --------------------------------------------------------
    # 2. Seeded parameters shared by all MPI ranks
    # --------------------------------------------------------

    rng = np.random.default_rng(seed)

    pulse_count = 3

    pulse_centers_x = rng.uniform(0.15, 0.85, pulse_count)
    pulse_centers_y = rng.uniform(0.15, 0.85, pulse_count)
    pulse_amplitudes = rng.uniform(0.5, 1.0, pulse_count)
    pulse_widths = rng.uniform(0.045, 0.09, pulse_count)

    # --------------------------------------------------------
    # 3. Heterogeneous diffusivity: 0.08 <= kappa <= 0.24
    # --------------------------------------------------------

    kappa_raw = (
        0.16
        + 0.035 * np.sin(2.0 * np.pi * X)
        * np.cos(2.0 * np.pi * Y)
        + 0.025 * np.sin(4.0 * np.pi * X + 0.5)
        * np.sin(2.0 * np.pi * Y)
    )

    kappa = np.clip(kappa_raw, 0.08, 0.24).astype(np.float64)

    # --------------------------------------------------------
    # 4. Initial condition: seeded Gaussian thermal pulses
    # --------------------------------------------------------

    u = np.zeros((LOCAL_NY + 2, LOCAL_NX + 2), dtype=np.float64)

    for px, py, amplitude, sigma in zip(
        pulse_centers_x,
        pulse_centers_y,
        pulse_amplitudes,
        pulse_widths
    ):
        distance_squared = (X - px)**2 + (Y - py)**2

        u += amplitude * np.exp(
            -distance_squared / (2.0 * sigma**2)
        )

    # --------------------------------------------------------
    # 5. Localized Gaussian source field
    # --------------------------------------------------------

    source_center_x = rng.uniform(0.25, 0.75)
    source_center_y = rng.uniform(0.25, 0.75)
    source_amplitude = rng.uniform(0.1, 0.5)
    source_sigma = 0.06

    source_distance_squared = (
        (X - source_center_x)**2
        + (Y - source_center_y)**2
    )

    source = source_amplitude * np.exp(
        -source_distance_squared / (2.0 * source_sigma**2)
    )

    source = source.astype(np.float64)

    # --------------------------------------------------------
    # 6. Enforce homogeneous Dirichlet physical boundaries
    # --------------------------------------------------------

    if nbr_up == MPI.PROC_NULL:
        u[0, :] = 0.0

    if nbr_down == MPI.PROC_NULL:
        u[-1, :] = 0.0

    if nbr_left == MPI.PROC_NULL:
        u[:, 0] = 0.0

    if nbr_right == MPI.PROC_NULL:
        u[:, -1] = 0.0

    # A second solution buffer for later time stepping
    u_new = u.copy()

    return {
        "X": X,
        "Y": Y,
        "u": u,
        "u_new": u_new,
        "kappa": kappa,
        "source": source,
        "dx": DX,
        "dy": DY,
        "pulse_centers": list(zip(
            pulse_centers_x.tolist(),
            pulse_centers_y.tolist()
        )),
        "source_center": (
            float(source_center_x),
            float(source_center_y)
        )
    }


'''

file_path = "distributed_cuda_solver.py"

with open(file_path, "r", encoding="utf-8") as f:
    original = f.read()

marker = "# TASK 1: INITIAL EXECUTION TEST"

if "def initialize_subdomains(context):" in original:
    print("Task 2 function already exists. No changes made.")
else:
    assert marker in original, "Could not find insertion point!"
    updated = original.replace(marker, task2_code + "\n" + marker, 1)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(updated)

    print("Task 2 function successfully added.")

Task 2 function successfully added.


In [25]:
test_code = r'''
import numpy as np
from mpi4py import MPI
from distributed_cuda_solver import (
    init_distributed_context,
    initialize_subdomains
)

context = init_distributed_context("comepotyyy")
data = initialize_subdomains(context)

rank = context["rank"]

# Check array dimensions
expected_shape = (258, 258)

for name in ("u", "u_new", "kappa", "source", "X", "Y"):
    assert data[name].shape == expected_shape, (
        f"Rank {rank}: incorrect shape for {name}"
    )

# Check numerical validity
assert np.all(np.isfinite(data["u"]))
assert np.all(np.isfinite(data["kappa"]))
assert np.all(np.isfinite(data["source"]))

# Check diffusivity range
assert np.min(data["kappa"]) >= 0.08
assert np.max(data["kappa"]) <= 0.24

# Check physical boundary ghost cells
if context["nbr_up"] == MPI.PROC_NULL:
    assert np.all(data["u"][0, :] == 0.0)

if context["nbr_down"] == MPI.PROC_NULL:
    assert np.all(data["u"][-1, :] == 0.0)

if context["nbr_left"] == MPI.PROC_NULL:
    assert np.all(data["u"][:, 0] == 0.0)

if context["nbr_right"] == MPI.PROC_NULL:
    assert np.all(data["u"][:, -1] == 0.0)

print(
    f"[Rank {rank}] TASK 2 PASS | "
    f"shape={data['u'].shape} | "
    f"kappa=[{data['kappa'].min():.6f}, "
    f"{data['kappa'].max():.6f}] | "
    f"u_max={data['u'].max():.6f} | "
    f"source_max={data['source'].max():.6f}",
    flush=True
)

context["cart_comm"].Barrier()

if rank == 0:
    print("TASK 2 VERIFICATION COMPLETED", flush=True)
'''

with open("test_task2.py", "w", encoding="utf-8") as f:
    f.write(test_code)

print("Task 2 test script created.")

Task 2 test script created.


In [26]:
import os
import sys
import subprocess

clean_env = os.environ.copy()

for key in list(clean_env.keys()):
    if key.startswith(("OMPI_", "PMIX_", "ORTE_", "OPAL_", "PRTE_")):
        clean_env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "ess", "hnp",
    "--tag-output",
    "-np", "4",
    sys.executable,
    "test_task2.py"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=120,
    env=clean_env
)

print("Return code:", result.returncode)
print("\nSTDOUT:\n", result.stdout or "(empty)")
print("\nSTDERR:\n", result.stderr or "(empty)")

if result.returncode != 0:
    raise RuntimeError("Task 2 verification failed. Send me the complete output.")

print("\nAll Task 2 checks passed.")

Return code: 0

STDOUT:
 [1,0]<stdout>:[Rank 0] Coordinates=[0, 0], Up=-2, Down=2, Left=-2, Right=1, GPU=0, Seed=7843665258511339110
[1,1]<stdout>:[Rank 1] Coordinates=[0, 1], Up=-2, Down=3, Left=0, Right=-2, GPU=0, Seed=7843665258511339110
[1,3]<stdout>:[Rank 3] Coordinates=[1, 1], Up=1, Down=-2, Left=2, Right=-2, GPU=0, Seed=7843665258511339110
[1,2]<stdout>:[Rank 2] Coordinates=[1, 0], Up=0, Down=-2, Left=-2, Right=3, GPU=0, Seed=7843665258511339110
[1,0]<stdout>:[Rank 0] TASK 2 PASS | shape=(258, 258) | kappa=[0.119700, 0.195148] | u_max=0.808698 | source_max=0.244212
[1,2]<stdout>:[Rank 2] TASK 2 PASS | shape=(258, 258) | kappa=[0.124777, 0.200300] | u_max=0.805607 | source_max=0.000830
[1,3]<stdout>:[Rank 3] TASK 2 PASS | shape=(258, 258) | kappa=[0.125146, 0.200301] | u_max=0.893285 | source_max=0.000830
[1,1]<stdout>:[Rank 1] TASK 2 PASS | shape=(258, 258) | kappa=[0.119699, 0.194927] | u_max=0.856197 | source_max=0.243963
[1,0]<stdout>:TASK 2 VERIFICATION COMPLETED


STDERR:
 

In [27]:

task3_kernels = r'''
# ============================================================
# TASK 3.1: GPU HALO PACK AND UNPACK KERNELS
# ============================================================

@cuda.jit
def pack_halos_kernel(u, packed, ny, nx, side):
    """
    Pack one owned border into a contiguous 1D device vector.

    side: 0=top, 1=bottom, 2=left, 3=right
    """
    i = cuda.grid(1)

    if side == 0 and i < nx:
        packed[i] = u[1, i + 1]

    elif side == 1 and i < nx:
        packed[i] = u[ny, i + 1]

    elif side == 2 and i < ny:
        packed[i] = u[i + 1, 1]

    elif side == 3 and i < ny:
        packed[i] = u[i + 1, nx]


@cuda.jit
def unpack_halos_kernel(packed, u, ny, nx, side):
    """
    Unpack a received 1D vector into the corresponding ghost boundary.

    side: 0=top ghost, 1=bottom ghost,
          2=left ghost, 3=right ghost
    """
    i = cuda.grid(1)

    if side == 0 and i < nx:
        u[0, i + 1] = packed[i]

    elif side == 1 and i < nx:
        u[ny + 1, i + 1] = packed[i]

    elif side == 2 and i < ny:
        u[i + 1, 0] = packed[i]

    elif side == 3 and i < ny:
        u[i + 1, nx + 1] = packed[i]
'''

file_path = "distributed_cuda_solver.py"

with open(file_path, "r", encoding="utf-8") as f:
    original = f.read()

marker = "def initialize_subdomains(context):"

if "def pack_halos_kernel(" in original:
    print("Task 3.1 kernels already exist. No changes made.")
else:
    assert marker in original, "Could not find Task 2 insertion point."

    updated = original.replace(
        marker,
        task3_kernels + "\n\n" + marker,
        1
    )

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(updated)

    print("Task 3.1 kernels successfully added.")
    print("Existing Task 1 and Task 2 code was preserved.")


Task 3.1 kernels successfully added.
Existing Task 1 and Task 2 code was preserved.


In [28]:

test_code = r'''
import numpy as np
from numba import cuda
from distributed_cuda_solver import (
    pack_halos_kernel,
    unpack_halos_kernel
)

ny = 256
nx = 256

# Build a test array with recognizable values.
host_u = np.zeros((ny + 2, nx + 2), dtype=np.float64)

host_u[1, 1:nx+1] = 10.0       # Top owned row
host_u[ny, 1:nx+1] = 20.0     # Bottom owned row
host_u[1:ny+1, 1] = 30.0      # Left owned column
host_u[1:ny+1, nx] = 40.0     # Right owned column

device_u = cuda.to_device(host_u)

threads = 256
blocks = (max(nx, ny) + threads - 1) // threads

# Check packing.
for side, expected in [
    (0, 10.0),
    (1, 20.0),
    (2, 30.0),
    (3, 40.0)
]:
    length = nx if side < 2 else ny

    device_packed = cuda.device_array(length, dtype=np.float64)

    pack_halos_kernel[blocks, threads](
        device_u, device_packed, ny, nx, side
    )

    packed = device_packed.copy_to_host()

    assert packed.shape == (length,)
    assert np.all(packed == expected), (
        f"Pack failed for side {side}"
    )

    # Put recognizable received data into a device vector.
    received = np.full(length, expected + 100.0, dtype=np.float64)
    device_received = cuda.to_device(received)

    # Clear the destination ghost boundary first.
    host_check = device_u.copy_to_host()

    if side == 0:
        host_check[0, :] = 0.0
    elif side == 1:
        host_check[ny+1, :] = 0.0
    elif side == 2:
        host_check[:, 0] = 0.0
    else:
        host_check[:, nx+1] = 0.0

    device_u = cuda.to_device(host_check)

    unpack_halos_kernel[blocks, threads](
        device_received, device_u, ny, nx, side
    )

    result = device_u.copy_to_host()

    if side == 0:
        actual = result[0, 1:nx+1]
    elif side == 1:
        actual = result[ny+1, 1:nx+1]
    elif side == 2:
        actual = result[1:ny+1, 0]
    else:
        actual = result[1:ny+1, nx+1]

    assert np.all(actual == expected + 100.0), (
        f"Unpack failed for side {side}"
    )

    print(f"Side {side}: PACK PASS | UNPACK PASS")

print("TASK 3.1 KERNEL VERIFICATION COMPLETED")
'''

with open("test_task3_kernels.py", "w", encoding="utf-8") as f:
    f.write(test_code)

print("Kernel test script created.")


Kernel test script created.


In [29]:
!python test_task3_kernels.py

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
Traceback (most recent call last):
  File "/content/test_task3_kernels.py", line 43, in <module>
    assert np.all(packed == expected), (
           ~~~~~~^^^^^^^^^^^^^^^^^^^^
AssertionError: Pack failed for side 0


In [30]:

test_code = r'''
import numpy as np
from numba import cuda
from distributed_cuda_solver import (
    pack_halos_kernel,
    unpack_halos_kernel
)

ny = 256
nx = 256
threads = 64
blocks = (max(nx, ny) + threads - 1) // threads

for side, expected in [
    (0, 10.0),
    (1, 20.0),
    (2, 30.0),
    (3, 40.0)
]:
    # Create a fresh array for each direction.
    host_u = np.zeros((ny + 2, nx + 2), dtype=np.float64)

    if side == 0:
        host_u[1, 1:nx+1] = expected
        length = nx
    elif side == 1:
        host_u[ny, 1:nx+1] = expected
        length = nx
    elif side == 2:
        host_u[1:ny+1, 1] = expected
        length = ny
    else:
        host_u[1:ny+1, nx] = expected
        length = ny

    device_u = cuda.to_device(host_u)
    device_packed = cuda.device_array(length, dtype=np.float64)

    # Test packing.
    pack_halos_kernel[blocks, threads](
        device_u, device_packed, ny, nx, side
    )

    packed = device_packed.copy_to_host()

    assert np.all(packed == expected), (
        f"Pack failed for side {side}: "
        f"min={packed.min()}, max={packed.max()}"
    )

    # Test unpacking with a different recognizable value.
    received_value = expected + 100.0
    received = np.full(length, received_value, dtype=np.float64)
    device_received = cuda.to_device(received)

    unpack_halos_kernel[blocks, threads](
        device_received, device_u, ny, nx, side
    )

    result = device_u.copy_to_host()

    if side == 0:
        actual = result[0, 1:nx+1]
    elif side == 1:
        actual = result[ny+1, 1:nx+1]
    elif side == 2:
        actual = result[1:ny+1, 0]
    else:
        actual = result[1:ny+1, nx+1]

    assert np.all(actual == received_value), (
        f"Unpack failed for side {side}"
    )

    print(f"Side {side}: PACK PASS | UNPACK PASS")

print("TASK 3.1 KERNEL VERIFICATION COMPLETED")
'''

with open("test_task3_kernels.py", "w", encoding="utf-8") as f:
    f.write(test_code)

print("Corrected test script saved.")


Corrected test script saved.


In [31]:
!python test_task3_kernels.py

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
Side 0: PACK PASS | UNPACK PASS
Side 1: PACK PASS | UNPACK PASS
Side 2: PACK PASS | UNPACK PASS
Side 3: PACK PASS | UNPACK PASS
TASK 3.1 KERNEL VERIFICATION COMPLETED


In [32]:

from pathlib import Path

path = Path("distributed_cuda_solver.py")

assert path.exists(), "distributed_cuda_solver.py was not found!"

code = path.read_text()

print("========== TASK 4 FUNCTION CHECK ==========")

for name in ["tiled_stencil_kernel", "step_overlapped"]:
    print(f"\n{name}:")
    if f"def {name}" in code:
        print("FOUND")
    else:
        print("NOT FOUND")

print("\n========== CUDA KERNEL AND FUNCTION LOCATIONS ==========")

lines = code.splitlines()

for i, line in enumerate(lines):
    if (
        "tiled_stencil_kernel" in line
        or "step_overlapped" in line
        or "SHARED_DIM" in line
        or "TILE_DIM" in line
    ):
        print(f"Line {i + 1}: {line}")


========== TASK 4 FUNCTION CHECK ==========

tiled_stencil_kernel:
NOT FOUND

step_overlapped:
NOT FOUND

========== CUDA KERNEL AND FUNCTION LOCATIONS ==========
Line 17: TILE_DIM = 16
Line 18: SHARED_DIM = TILE_DIM + 2


In [33]:
TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2

In [34]:

from pathlib import Path

code = Path("distributed_cuda_solver.py").read_text()
lines = code.splitlines()

print("========== CONSTANTS AND CONFIGURATION ==========")

for i, line in enumerate(lines[:100]):
    print(f"{i+1:03}: {line}")

print("\n========== INITIALIZATION FUNCTION ==========")

start = next(
    (i for i, line in enumerate(lines)
     if line.startswith("def initialize_subdomains")),
    None
)

if start is None:
    print("initialize_subdomains NOT FOUND")
else:
    end = next(
        (i for i in range(start + 1, len(lines))
         if lines[i].startswith("def ") or lines[i].startswith("@cuda.jit")),
        len(lines)
    )

    for i in range(start, end):
        print(f"{i+1:03}: {lines[i]}")


========== CONSTANTS AND CONFIGURATION ==========
001: 
002: import os
003: import sys
004: import json
005: import math
006: import hashlib
007: import subprocess
008: 
009: import numpy as np
010: from mpi4py import MPI
011: from numba import cuda, float64
012: 
013: # ============================================================
014: # GLOBAL CONFIGURATION
015: # ============================================================
016: 
017: TILE_DIM = 16
018: SHARED_DIM = TILE_DIM + 2
019: 
020: GLOBAL_NX = 512
021: GLOBAL_NY = 512
022: 
023: STEPS = 400
024: LAMBDA_DAMP = 0.15
025: 
026: 
027: # ============================================================
028: # HELPER 1: GET GPU UUID
029: # ============================================================
030: 
031: def get_gpu_uuid() -> str:
032:     """Get active NVIDIA GPU UUID."""
033: 
034:     out = subprocess.check_output(
035:         [
036:             "nvidia-smi",
037:             "--query-gpu=uuid",
038:             "--format=csv,n

In [35]:

from pathlib import Path

lines = Path("distributed_cuda_solver.py").read_text().splitlines()

print("========== LINES 95–210 ==========")

for i in range(94, min(210, len(lines))):
    print(f"{i+1:03}: {lines[i]}")

print("\n========== TIME-STEP REFERENCES ==========")

for i, line in enumerate(lines):
    if any(term in line.lower() for term in [
        "dt", "timestep", "time_step", "delta_t",
        "diffusion", "stencil"
    ]):
        print(f"Line {i+1}: {line}")


========== LINES 95–210 ==========
095: 
096:     # 5. Find vertical neighbors
097:     nbr_up, nbr_down = cart_comm.Shift(0, 1)
098: 
099:     # 6. Find horizontal neighbors
100:     nbr_left, nbr_right = cart_comm.Shift(1, 1)
101: 
102:     # 7. Bind process to GPU 0
103:     cuda.select_device(0)
104: 
105:     # 8. Obtain GPU UUID on rank 0
106:     gpu_uuid = get_gpu_uuid() if rank == 0 else None
107: 
108:     # Broadcast UUID to other ranks
109:     gpu_uuid = cart_comm.bcast(gpu_uuid, root=0)
110: 
111:     # 9. Derive personal seed
112:     instance_seed = derive_student_seed(
113:         github_user,
114:         gpu_uuid
115:     )
116: 
117:     # 10. Display information
118:     print(
119:         f"[Rank {rank}] "
120:         f"Coordinates={coords}, "
121:         f"Up={nbr_up}, "
122:         f"Down={nbr_down}, "
123:         f"Left={nbr_left}, "
124:         f"Right={nbr_right}, "
125:         f"GPU=0, "
126:         f"Seed={instance_seed}",
127:         flush=True
1

In [36]:
from pathlib import Path

code = Path("distributed_cuda_solver.py").read_text()

print("========== TASK 3.2 AND TASK 4 DEPENDENCIES ==========")

for name in [
    "exchange_halos_async",
    "tiled_stencil_kernel",
    "step_overlapped",
    "DT",
    "DT_MAX"
]:
    found = name in code
    print(f"{name}: {'FOUND' if found else 'NOT FOUND'}")

print("\n========== FILE END ==========")

for i, line in enumerate(code.splitlines()[-35:]):
    print(f"{len(code.splitlines()) - 34 + i:03}: {line}")


========== TASK 3.2 AND TASK 4 DEPENDENCIES ==========
exchange_halos_async: NOT FOUND
tiled_stencil_kernel: NOT FOUND
step_overlapped: NOT FOUND
DT: NOT FOUND
DT_MAX: NOT FOUND

========== FILE END ==========
331:         "X": X,
332:         "Y": Y,
333:         "u": u,
334:         "u_new": u_new,
335:         "kappa": kappa,
336:         "source": source,
337:         "dx": DX,
338:         "dy": DY,
339:         "pulse_centers": list(zip(
340:             pulse_centers_x.tolist(),
341:             pulse_centers_y.tolist()
342:         )),
343:         "source_center": (
344:             float(source_center_x),
345:             float(source_center_y)
346:         )
347:     }
348: 
349: 
350: 
351: # TASK 1: INITIAL EXECUTION TEST
352: # ============================================================
353: 
354: if __name__ == "__main__":
355: 
356:     github_user = sys.argv[1]
357: 
358:     context = init_distributed_context(github_user)
359: 
360:     if context["rank"] == 0:
361: 

In [37]:
    elif side == 3 and i < ny:
        u[i + 1, nx + 1] = packed[i]

SyntaxError: invalid syntax (3478177814.py, line 1)

In [38]:

from pathlib import Path

lines = Path("distributed_cuda_solver.py").read_text().splitlines()

for i, line in enumerate(lines):
    if "def unpack_halos_kernel" in line:
        start = max(0, i - 2)
        end = min(len(lines), i + 65)

        for j in range(start, end):
            print(f"{j+1:03}: {lines[j]}")

        break


186: 
187: @cuda.jit
188: def unpack_halos_kernel(packed, u, ny, nx, side):
189:     """
190:     Unpack a received 1D vector into the corresponding ghost boundary.
191: 
192:     side: 0=top ghost, 1=bottom ghost,
193:           2=left ghost, 3=right ghost
194:     """
195:     i = cuda.grid(1)
196: 
197:     if side == 0 and i < nx:
198:         u[0, i + 1] = packed[i]
199: 
200:     elif side == 1 and i < nx:
201:         u[ny + 1, i + 1] = packed[i]
202: 
203:     elif side == 2 and i < ny:
204:         u[i + 1, 0] = packed[i]
205: 
206:     elif side == 3 and i < ny:
207:         u[i + 1, nx + 1] = packed[i]
208: 
209: 
210: def initialize_subdomains(context):
211:     """
212:     Create rank-local initial fields with one ghost-cell layer.
213: 
214:     Array indexing:
215:         axis 0 = y / rows
216:         axis 1 = x / columns
217: 
218:     Owned cells: [1:257, 1:257]
219:     Ghost cells: index 0 and index 257
220:     """
221: 
222:     row_rank = context["row_rank"]
22

In [39]:

from pathlib import Path
import ast

path = Path("distributed_cuda_solver.py")
source = path.read_text()

if "def exchange_halos_async(" in source:
    print("exchange_halos_async already exists. No insertion needed.")

else:
    marker = "\ndef initialize_subdomains(context):"

    assert marker in source, (
        "Could not find initialize_subdomains. File was not modified."
    )

    new_function = '''

# ============================================================
# TASK 3.2: ASYNCHRONOUS MPI HALO EXCHANGE
# ============================================================

def exchange_halos_async(
    u_device, context, ny, nx,
    stream_boundary, interior_callback=None
):
    """Exchange GPU halos using pinned memory and non-blocking MPI."""

    cart_comm = context["cart_comm"]

    neighbors = {
        0: context["nbr_up"],
        1: context["nbr_down"],
        2: context["nbr_left"],
        3: context["nbr_right"]
    }

    send_tags = {0: 10, 1: 11, 2: 12, 3: 13}
    receive_tags = {0: 11, 1: 10, 2: 13, 3: 12}

    lengths = {0: nx, 1: nx, 2: ny, 3: ny}

    device_send = {}
    device_recv = {}
    host_send = {}
    host_recv = {}

    for side in range(4):
        count = lengths[side]

        device_send[side] = cuda.device_array(
            count, dtype=np.float64
        )
        device_recv[side] = cuda.device_array(
            count, dtype=np.float64
        )

        host_send[side] = cuda.pinned_array(
            count, dtype=np.float64
        )
        host_recv[side] = cuda.pinned_array(
            count, dtype=np.float64
        )

    # Stage A: pack borders and copy them to pinned host memory.
    for side in range(4):
        count = lengths[side]
        blocks = (count + 255) // 256

        pack_halos_kernel[blocks, 256, stream_boundary](
            u_device, device_send[side], ny, nx, side
        )

        device_send[side].copy_to_host(
            ary=host_send[side],
            stream=stream_boundary
        )

    stream_boundary.synchronize()

    # Stage B: post receives first, then sends.
    requests = []

    for side in range(4):
        neighbor = neighbors[side]

        if neighbor != MPI.PROC_NULL:
            requests.append(
                cart_comm.Irecv(
                    host_recv[side],
                    source=neighbor,
                    tag=receive_tags[side]
                )
            )

    for side in range(4):
        neighbor = neighbors[side]

        if neighbor != MPI.PROC_NULL:
            requests.append(
                cart_comm.Isend(
                    host_send[side],
                    dest=neighbor,
                    tag=send_tags[side]
                )
            )

    # The caller may launch independent interior GPU work here.
    if interior_callback is not None:
        interior_callback()

    # Stage C: finish communication and unpack received halos.
    if requests:
        MPI.Request.Waitall(requests)

    for side in range(4):
        if neighbors[side] == MPI.PROC_NULL:
            continue

        count = lengths[side]
        blocks = (count + 255) // 256

        device_recv[side].copy_to_device(
            host_recv[side],
            stream=stream_boundary
        )

        unpack_halos_kernel[blocks, 256, stream_boundary](
            device_recv[side], u_device, ny, nx, side
        )

    stream_boundary.synchronize()

    return {
        "host_send": host_send,
        "host_recv": host_recv,
        "device_send": device_send,
        "device_recv": device_recv
    }

'''

    source = source.replace(
        marker,
        "\n" + new_function + marker,
        1
    )

    # Validate before saving.
    ast.parse(source)

    path.write_text(source)

    print("Task 3.2 function inserted successfully.")

# Final validation.
source = path.read_text()
ast.parse(source)

assert "def exchange_halos_async(" in source

print("TASK 3.2 SOURCE CHECK: PASS")
print("Python syntax: VALID")


Task 3.2 function inserted successfully.
TASK 3.2 SOURCE CHECK: PASS
Python syntax: VALID


In [40]:

from pathlib import Path
import os
import sys
import subprocess

test_code = r'''
import sys
import numpy as np
from mpi4py import MPI
from numba import cuda

from distributed_cuda_solver import (
    init_distributed_context,
    exchange_halos_async,
    LOCAL_NX,
    LOCAL_NY
)

github_user = sys.argv[1]

context = init_distributed_context(github_user)
rank = context["rank"]
cart_comm = context["cart_comm"]

ny = LOCAL_NY
nx = LOCAL_NX

# Give every rank a unique value in its owned cells.
host_u = np.zeros((ny + 2, nx + 2), dtype=np.float64)
host_u[1:ny+1, 1:nx+1] = float(rank + 1)

u_device = cuda.to_device(host_u)
stream_boundary = cuda.stream()

# Perform the actual asynchronous MPI halo exchange.
exchange_halos_async(
    u_device=u_device,
    context=context,
    ny=ny,
    nx=nx,
    stream_boundary=stream_boundary
)

result = u_device.copy_to_host()

# Verify received halos against the neighboring rank.
checks = [
    ("top", context["nbr_up"], result[0, 1:-1]),
    ("bottom", context["nbr_down"], result[-1, 1:-1]),
    ("left", context["nbr_left"], result[1:-1, 0]),
    ("right", context["nbr_right"], result[1:-1, -1])
]

local_pass = True

for direction, neighbor, actual in checks:
    if neighbor == MPI.PROC_NULL:
        expected = 0.0
    else:
        expected = float(neighbor + 1)

    if not np.all(actual == expected):
        local_pass = False
        print(
            f"[Rank {rank}] FAIL {direction}: "
            f"expected {expected}, "
            f"actual min={actual.min()}, max={actual.max()}",
            flush=True
        )

# Check all physical boundary ghost cells remain zero.
if context["nbr_up"] == MPI.PROC_NULL:
    local_pass &= np.all(result[0, :] == 0.0)

if context["nbr_down"] == MPI.PROC_NULL:
    local_pass &= np.all(result[-1, :] == 0.0)

if context["nbr_left"] == MPI.PROC_NULL:
    local_pass &= np.all(result[:, 0] == 0.0)

if context["nbr_right"] == MPI.PROC_NULL:
    local_pass &= np.all(result[:, -1] == 0.0)

# Collect verification status from all ranks.
global_pass = cart_comm.allreduce(
    int(local_pass),
    op=MPI.MIN
)

print(
    f"[Rank {rank}] "
    f"{'TASK 3.2 PASS' if local_pass else 'TASK 3.2 FAIL'}",
    flush=True
)

cart_comm.Barrier()

if rank == 0:
    if global_pass:
        print("TASK 3.2 FOUR-RANK VERIFICATION PASSED", flush=True)
    else:
        print("TASK 3.2 FOUR-RANK VERIFICATION FAILED", flush=True)

if not global_pass:
    raise RuntimeError("At least one MPI rank failed halo verification")
'''

Path("test_task32.py").write_text(test_code)

# Remove inherited MPI environment variables from the notebook process.
clean_env = os.environ.copy()

for key in list(clean_env.keys()):
    if key.startswith(("OMPI_", "PMIX_", "ORTE_", "OPAL_", "PRTE_")):
        clean_env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "ess", "hnp",
    "--tag-output",
    "-np", "4",
    sys.executable,
    "test_task32.py",
    "comepotyyy"
]

print("Launching Task 3.2 verification...\n")

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=180,
    env=clean_env
)

print("Return code:", result.returncode)
print("\n========== STDOUT ==========")
print(result.stdout if result.stdout else "(empty)")

print("\n========== STDERR ==========")
print(result.stderr if result.stderr else "(empty)")


Launching Task 3.2 verification...

Return code: 0

========== STDOUT ==========
[1,0]<stdout>:[Rank 0] Coordinates=[0, 0], Up=-2, Down=2, Left=-2, Right=1, GPU=0, Seed=7843665258511339110
[1,1]<stdout>:[Rank 1] Coordinates=[0, 1], Up=-2, Down=3, Left=0, Right=-2, GPU=0, Seed=7843665258511339110
[1,3]<stdout>:[Rank 3] Coordinates=[1, 1], Up=1, Down=-2, Left=2, Right=-2, GPU=0, Seed=7843665258511339110
[1,2]<stdout>:[Rank 2] Coordinates=[1, 0], Up=0, Down=-2, Left=-2, Right=3, GPU=0, Seed=7843665258511339110
[1,2]<stdout>:[Rank 2] TASK 3.2 PASS
[1,0]<stdout>:[Rank 0] TASK 3.2 PASS
[1,1]<stdout>:[Rank 1] TASK 3.2 PASS
[1,3]<stdout>:[Rank 3] TASK 3.2 PASS
[1,0]<stdout>:TASK 3.2 FOUR-RANK VERIFICATION PASSED


========== STDERR ==========
[1,3]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
[1,3]<stderr>:  warn(errors.NumbaPerformanceWarning(msg))

In [41]:

from pathlib import Path
import ast

path = Path("distributed_cuda_solver.py")
source = path.read_text()

if "def tiled_stencil_kernel(" in source:
    print("Task 4 kernel already exists. No insertion performed.")

else:
    marker = "\ndef initialize_subdomains(context):"

    assert marker in source, "Insertion point not found."

    task4_code = '''

# ============================================================
# TASK 4: SHARED-MEMORY TILED STENCIL
# ============================================================

@cuda.jit
def tiled_stencil_kernel(
    u, u_new, kappa, source,
    ny, nx, dx, dy, dt,
    r_start, r_end, c_start, c_end,
    boundary_only
):
    """
    Five-point anisotropic reaction-diffusion stencil.

    Arrays use [row, column] = [y, x].
    Bounding boxes use exclusive upper bounds.

    boundary_only=False: update cells inside the bounding box.
    boundary_only=True: update only the outer owned-cell ring.
    """

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    # Shared-memory tile including a one-cell halo apron.
    smem = cuda.shared.array(
        shape=(SHARED_DIM, SHARED_DIM),
        dtype=float64
    )

    # Global indices of the owned cell for this thread.
    col = cuda.blockIdx.x * TILE_DIM + tx + 1
    row = cuda.blockIdx.y * TILE_DIM + ty + 1

    sx = tx + 1
    sy = ty + 1

    # Every thread loads its center cell.
    smem[sy, sx] = u[row, col]

    # Load left and right apron.
    if tx == 0:
        smem[sy, 0] = u[row, col - 1]

    if tx == TILE_DIM - 1:
        smem[sy, TILE_DIM + 1] = u[row, col + 1]

    # Load top and bottom apron.
    if ty == 0:
        smem[0, sx] = u[row - 1, col]

    if ty == TILE_DIM - 1:
        smem[TILE_DIM + 1, sx] = u[row + 1, col]

    # All apron and center loads must finish before stencil reads.
    cuda.syncthreads()

    # Check the requested rectangular bounding box.
    inside_box = (
        row >= r_start and row < r_end
        and col >= c_start and col < c_end
    )

    # For the boundary launch, select only the outer owned ring.
    on_boundary = (
        row == 1 or row == ny
        or col == 1 or col == nx
    )

    if inside_box and (not boundary_only or on_boundary):

        center = smem[sy, sx]

        north = smem[sy - 1, sx]
        south = smem[sy + 1, sx]
        west = smem[sy, sx - 1]
        east = smem[sy, sx + 1]

        # Arithmetic-average interface diffusivities.
        k_north = 0.5 * (
            kappa[row - 1, col] + kappa[row, col]
        )

        k_south = 0.5 * (
            kappa[row + 1, col] + kappa[row, col]
        )

        k_west = 0.5 * (
            kappa[row, col - 1] + kappa[row, col]
        )

        k_east = 0.5 * (
            kappa[row, col + 1] + kappa[row, col]
        )

        # Diffusion in the y direction (rows).
        diffusion_y = (
            k_south * (south - center)
            - k_north * (center - north)
        ) / (dy * dy)

        # Diffusion in the x direction (columns).
        diffusion_x = (
            k_east * (east - center)
            - k_west * (center - west)
        ) / (dx * dx)

        # Nonlinear reaction and localized source.
        reaction = -LAMBDA_DAMP * center * center * center

        rhs = (
            diffusion_x
            + diffusion_y
            + reaction
            + source[row, col]
        )

        # Explicit Euler update.
        u_new[row, col] = center + dt * rhs


# ============================================================
# TASK 4: DUAL-STREAM OVERLAPPED TIME STEP
# ============================================================

def step_overlapped(
    u_device,
    u_new_device,
    kappa_device,
    source_device,
    context,
    dt,
    dx,
    dy,
    stream_boundary,
    stream_interior
):
    """
    One overlapped time step.

    1. Exchange halos while computing deep interior cells.
    2. Compute boundary cells after halo reception.
    3. Synchronize both streams before returning.
    """

    ny = LOCAL_NY
    nx = LOCAL_NX

    threads = (TILE_DIM, TILE_DIM)
    blocks = (
        (nx + TILE_DIM - 1) // TILE_DIM,
        (ny + TILE_DIM - 1) // TILE_DIM
    )

    # Stage B callback: launch deep-interior stencil while MPI runs.
    def compute_deep_interior():

        tiled_stencil_kernel[blocks, threads, stream_interior](
            u_device,
            u_new_device,
            kappa_device,
            source_device,
            ny, nx, dx, dy, dt,
            2, ny,
            2, nx,
            False
        )

    # Stages A and B: pack, start MPI, and launch interior work.
    exchange_result = exchange_halos_async(
        u_device=u_device,
        context=context,
        ny=ny,
        nx=nx,
        stream_boundary=stream_boundary,
        interior_callback=compute_deep_interior
    )

    # Stage C: halo data is now available.
    # Compute only the outer owned-cell ring.
    tiled_stencil_kernel[blocks, threads, stream_boundary](
        u_device,
        u_new_device,
        kappa_device,
        source_device,
        ny, nx, dx, dy, dt,
        1, ny + 1,
        1, nx + 1,
        True
    )

    # Both regions must finish before the buffers can be swapped.
    stream_interior.synchronize()
    stream_boundary.synchronize()

    return exchange_result

'''

    source = source.replace(
        marker,
        "\n" + task4_code + marker,
        1
    )

    ast.parse(source)
    path.write_text(source)

    print("Task 4 implementation inserted.")

# Final source validation.
source = path.read_text()
ast.parse(source)

assert "def tiled_stencil_kernel(" in source
assert "def step_overlapped(" in source
assert "cuda.syncthreads()" in source

print("TASK 4 SOURCE CHECK: PASS")
print("tiled_stencil_kernel: FOUND")
print("step_overlapped: FOUND")
print("Shared-memory synchronization: FOUND")
print("Python syntax: VALID")


Task 4 implementation inserted.
TASK 4 SOURCE CHECK: PASS
tiled_stencil_kernel: FOUND
step_overlapped: FOUND
Shared-memory synchronization: FOUND
Python syntax: VALID


In [42]:

import numpy as np
from numba import cuda

from distributed_cuda_solver import (
    tiled_stencil_kernel,
    TILE_DIM,
    LAMBDA_DAMP
)

# --------------------------------------------------
# 1. Prepare test arrays
# --------------------------------------------------

ny = 32
nx = 32

dx = 1.0 / 33.0
dy = 1.0 / 33.0
dt = 1.0e-7

rng = np.random.default_rng(42)

u = np.zeros((ny + 2, nx + 2), dtype=np.float64)
u[1:ny+1, 1:nx+1] = rng.uniform(0.1, 0.8, (ny, nx))

kappa = rng.uniform(
    0.08, 0.24, (ny + 2, nx + 2)
).astype(np.float64)

source = rng.uniform(
    0.0, 0.1, (ny + 2, nx + 2)
).astype(np.float64)

# Keep all physical ghost values at zero for this test.
u[0, :] = 0.0
u[-1, :] = 0.0
u[:, 0] = 0.0
u[:, -1] = 0.0

# --------------------------------------------------
# 2. Independent CPU reference
# --------------------------------------------------

cpu_result = u.copy()

for i in range(1, ny + 1):
    for j in range(1, nx + 1):

        center = u[i, j]

        k_north = 0.5 * (kappa[i-1, j] + kappa[i, j])
        k_south = 0.5 * (kappa[i+1, j] + kappa[i, j])
        k_west = 0.5 * (kappa[i, j-1] + kappa[i, j])
        k_east = 0.5 * (kappa[i, j+1] + kappa[i, j])

        diffusion_y = (
            k_south * (u[i+1, j] - center)
            - k_north * (center - u[i-1, j])
        ) / dy**2

        diffusion_x = (
            k_east * (u[i, j+1] - center)
            - k_west * (center - u[i, j-1])
        ) / dx**2

        reaction = -LAMBDA_DAMP * center**3

        rhs = (
            diffusion_x
            + diffusion_y
            + reaction
            + source[i, j]
        )

        cpu_result[i, j] = center + dt * rhs

# --------------------------------------------------
# 3. Execute CUDA tiled kernel
# --------------------------------------------------

u_gpu = cuda.to_device(u)
u_new_gpu = cuda.to_device(np.zeros_like(u))
kappa_gpu = cuda.to_device(kappa)
source_gpu = cuda.to_device(source)

threads = (TILE_DIM, TILE_DIM)
blocks = (
    (nx + TILE_DIM - 1) // TILE_DIM,
    (ny + TILE_DIM - 1) // TILE_DIM
)

tiled_stencil_kernel[blocks, threads](
    u_gpu,
    u_new_gpu,
    kappa_gpu,
    source_gpu,
    ny, nx, dx, dy, dt,
    1, ny + 1,
    1, nx + 1,
    False
)

gpu_result = u_new_gpu.copy_to_host()

# --------------------------------------------------
# 4. Compare CPU and GPU results
# --------------------------------------------------

difference = np.abs(cpu_result[1:-1, 1:-1] -
                    gpu_result[1:-1, 1:-1])

max_error = float(np.max(difference))

print("========== TASK 4 KERNEL VERIFICATION ==========")
print("Grid:", (ny, nx))
print("CUDA block:", threads)
print("Shared tile:", (TILE_DIM + 2, TILE_DIM + 2))
print("Maximum absolute error:", max_error)

if np.allclose(
    cpu_result[1:-1, 1:-1],
    gpu_result[1:-1, 1:-1],
    rtol=1e-12,
    atol=1e-12
):
    print("TASK 4 KERNEL NUMERICAL TEST: PASS")
else:
    print("TASK 4 KERNEL NUMERICAL TEST: FAIL")
    raise AssertionError("CPU and GPU stencil results differ")


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


========== TASK 4 KERNEL VERIFICATION ==========
Grid: (32, 32)
CUDA block: (16, 16)
Shared tile: (18, 18)
Maximum absolute error: 0.0
TASK 4 KERNEL NUMERICAL TEST: PASS


In [43]:

%%writefile test_task4_overlapped.py

import sys
import numpy as np
from mpi4py import MPI
from numba import cuda

from distributed_cuda_solver import (
    init_distributed_context,
    initialize_subdomains,
    step_overlapped,
    LOCAL_NX,
    LOCAL_NY,
    LAMBDA_DAMP
)

github_user = sys.argv[1]

# --------------------------------------------------
# 1. Initialize MPI and local fields
# --------------------------------------------------

context = init_distributed_context(github_user)
rank = context["rank"]
cart_comm = context["cart_comm"]

fields = initialize_subdomains(context)

u = fields["u"]
kappa = fields["kappa"]
source = fields["source"]

dx = fields["dx"]
dy = fields["dy"]

ny = LOCAL_NY
nx = LOCAL_NX

# Conservative test time step.
dt = 1.0e-7

# --------------------------------------------------
# 2. Build independent CPU reference
# --------------------------------------------------

cpu_reference = np.zeros_like(u)

for i in range(1, ny + 1):
    for j in range(1, nx + 1):

        center = u[i, j]

        k_north = 0.5 * (kappa[i-1, j] + kappa[i, j])
        k_south = 0.5 * (kappa[i+1, j] + kappa[i, j])
        k_west = 0.5 * (kappa[i, j-1] + kappa[i, j])
        k_east = 0.5 * (kappa[i, j+1] + kappa[i, j])

        diffusion_y = (
            k_south * (u[i+1, j] - center)
            - k_north * (center - u[i-1, j])
        ) / dy**2

        diffusion_x = (
            k_east * (u[i, j+1] - center)
            - k_west * (center - u[i, j-1])
        ) / dx**2

        reaction = -LAMBDA_DAMP * center**3

        rhs = (
            diffusion_x
            + diffusion_y
            + reaction
            + source[i, j]
        )

        cpu_reference[i, j] = center + dt * rhs

# --------------------------------------------------
# 3. Transfer arrays to GPU
# --------------------------------------------------

u_device = cuda.to_device(u)
u_new_device = cuda.to_device(np.zeros_like(u))
kappa_device = cuda.to_device(kappa)
source_device = cuda.to_device(source)

stream_boundary = cuda.stream()
stream_interior = cuda.stream()

# --------------------------------------------------
# 4. Execute one complete overlapped time step
# --------------------------------------------------

step_overlapped(
    u_device=u_device,
    u_new_device=u_new_device,
    kappa_device=kappa_device,
    source_device=source_device,
    context=context,
    dt=dt,
    dx=dx,
    dy=dy,
    stream_boundary=stream_boundary,
    stream_interior=stream_interior
)

gpu_result = u_new_device.copy_to_host()

# --------------------------------------------------
# 5. Compare every owned cell
# --------------------------------------------------

difference = np.abs(
    cpu_reference[1:ny+1, 1:nx+1]
    - gpu_result[1:ny+1, 1:nx+1]
)

local_error = float(np.max(difference))

global_error = cart_comm.allreduce(
    local_error,
    op=MPI.MAX
)

local_pass = np.allclose(
    cpu_reference[1:ny+1, 1:nx+1],
    gpu_result[1:ny+1, 1:nx+1],
    rtol=1e-12,
    atol=1e-12
)

global_pass = cart_comm.allreduce(
    int(local_pass),
    op=MPI.MIN
)

print(
    f"[Rank {rank}] "
    f"{'PASS' if local_pass else 'FAIL'} "
    f"| max_error={local_error:.16e}",
    flush=True
)

cart_comm.Barrier()

if rank == 0:
    print("Global maximum error:", f"{global_error:.16e}")

    if global_pass:
        print("TASK 4 OVERLAPPED MPI VERIFICATION: PASS")
    else:
        print("TASK 4 OVERLAPPED MPI VERIFICATION: FAIL")

if not global_pass:
    raise RuntimeError("Overlapped step differs from CPU reference")


Writing test_task4_overlapped.py


In [44]:

import os
import sys
import subprocess

clean_env = os.environ.copy()

for key in list(clean_env.keys()):
    if key.startswith(("OMPI_", "PMIX_", "ORTE_", "OPAL_", "PRTE_")):
        clean_env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "ess", "hnp",
    "--tag-output",
    "-np", "4",
    sys.executable,
    "test_task4_overlapped.py",
    "comepotyyy"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=240,
    env=clean_env
)

print("Return code:", result.returncode)
print("\n========== STDOUT ==========")
print(result.stdout or "(empty)")
print("\n========== STDERR ==========")
print(result.stderr or "(empty)")


Return code: 0

========== STDOUT ==========
[1,0]<stdout>:[Rank 0] Coordinates=[0, 0], Up=-2, Down=2, Left=-2, Right=1, GPU=0, Seed=7843665258511339110
[1,1]<stdout>:[Rank 1] Coordinates=[0, 1], Up=-2, Down=3, Left=0, Right=-2, GPU=0, Seed=7843665258511339110
[1,3]<stdout>:[Rank 3] Coordinates=[1, 1], Up=1, Down=-2, Left=2, Right=-2, GPU=0, Seed=7843665258511339110
[1,2]<stdout>:[Rank 2] Coordinates=[1, 0], Up=0, Down=-2, Left=-2, Right=3, GPU=0, Seed=7843665258511339110
[1,0]<stdout>:[Rank 0] PASS | max_error=1.3877787807814457e-17
[1,1]<stdout>:[Rank 1] PASS | max_error=2.1175823681357508e-22
[1,3]<stdout>:[Rank 3] PASS | max_error=1.7347234759768071e-18
[1,2]<stdout>:[Rank 2] PASS | max_error=1.7347234759768071e-18
[1,0]<stdout>:Global maximum error: 1.3877787807814457e-17
[1,0]<stdout>:TASK 4 OVERLAPPED MPI VERIFICATION: PASS


========== STDERR ==========
[1,0]<stderr>:/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid s

In [45]:

import re
import numpy as np
from numba import cuda

from distributed_cuda_solver import (
    tiled_stencil_kernel,
    TILE_DIM,
    LAMBDA_DAMP
)

# Compile the kernel using a small test grid.
ny = 32
nx = 32

dx = 1.0 / 33.0
dy = 1.0 / 33.0
dt = 1.0e-7

u = np.ones((ny + 2, nx + 2), dtype=np.float64)
u[0, :] = 0.0
u[-1, :] = 0.0
u[:, 0] = 0.0
u[:, -1] = 0.0

u_new = np.zeros_like(u)
kappa = np.full_like(u, 0.15)
source = np.zeros_like(u)

u_gpu = cuda.to_device(u)
u_new_gpu = cuda.to_device(u_new)
kappa_gpu = cuda.to_device(kappa)
source_gpu = cuda.to_device(source)

threads = (TILE_DIM, TILE_DIM)
blocks = (2, 2)

tiled_stencil_kernel[blocks, threads](
    u_gpu,
    u_new_gpu,
    kappa_gpu,
    source_gpu,
    ny,
    nx,
    dx,
    dy,
    dt,
    1,
    ny + 1,
    1,
    nx + 1,
    False
)

cuda.synchronize()

# Retrieve generated PTX assembly.
asm_result = tiled_stencil_kernel.inspect_asm()

if isinstance(asm_result, dict):
    ptx = "\n".join(asm_result.values())
else:
    ptx = str(asm_result)

# Count required PTX instructions.
store_count = len(re.findall(
    r"^\s*st\.shared\.f64\b", ptx, re.MULTILINE
))

load_count = len(re.findall(
    r"^\s*ld\.shared\.f64\b", ptx, re.MULTILINE
))

barrier_count = len(re.findall(
    r"^\s*bar\.sync\b", ptx, re.MULTILINE
))

print("========== TASK 4 PTX INSPECTION ==========")
print("PTX assembly retrieved:", bool(ptx))
print("st.shared.f64 count:", store_count)
print("ld.shared.f64 count:", load_count)
print("bar.sync count:", barrier_count)

print("\nShared-memory instruction lines:")
for line in ptx.splitlines():
    if re.search(r"\b(st|ld)\.shared\b|\bbar\.sync\b", line):
        print(line.strip())

print("\nPTX INSPECTION COMPLETED")


========== TASK 4 PTX INSPECTION ==========
PTX assembly retrieved: True
st.shared.f64 count: 5
ld.shared.f64 count: 5
bar.sync count: 1

Shared-memory instruction lines:
st.shared.f64 	[%rd44+152], %fd4;
st.shared.f64 	[%rd66+144], %fd5;
st.shared.f64 	[%rd88+280], %fd6;
st.shared.f64 	[%rd1], %fd7;
st.shared.f64 	[%rd1+2448], %fd8;
bar.sync 	0;
ld.shared.f64 	%fd22, [%rd170+296];
ld.shared.f64 	%fd23, [%rd158+8];
ld.shared.f64 	%fd26, [%rd164+8];
ld.shared.f64 	%fd32, [%rd180+16];
ld.shared.f64 	%fd35, [%rd175];

PTX INSPECTION COMPLETED


In [46]:

from pathlib import Path
import ast

path = Path("distributed_cuda_solver.py")
source_text = path.read_text()

if "def step_serialized(" in source_text or "def run_solver(" in source_text:
    raise RuntimeError(
        "Solver functions already exist. Do not insert duplicates."
    )

new_code = r'''

def step_serialized(
    u_device, u_new_device, kappa_device, source_device,
    context, dt, dx, dy, stream_boundary
):
    """Exchange halos completely, then compute the full owned domain."""
    ny = LOCAL_NY
    nx = LOCAL_NX

    exchange_halos_async(
        u_device, context, ny, nx, stream_boundary
    )

    threads = (TILE_DIM, TILE_DIM)
    blocks = (
        (nx + TILE_DIM - 1) // TILE_DIM,
        (ny + TILE_DIM - 1) // TILE_DIM
    )

    tiled_stencil_kernel[blocks, threads, stream_boundary](
        u_device, u_new_device, kappa_device, source_device,
        ny, nx, dx, dy, dt,
        1, ny + 1, 1, nx + 1, False
    )

    stream_boundary.synchronize()


def _run_one_mode(context, mode, steps, dt, fields):
    """Execute one complete simulation mode on one MPI rank."""
    cart_comm = context["cart_comm"]

    ny = LOCAL_NY
    nx = LOCAL_NX

    dx = fields["dx"]
    dy = fields["dy"]

    u_device = cuda.to_device(fields["u"])
    u_new_device = cuda.to_device(np.zeros_like(fields["u"]))
    kappa_device = cuda.to_device(fields["kappa"])
    source_device = cuda.to_device(fields["source"])

    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()

    def global_sum(value):
        send = np.array(value, dtype=np.float64)
        receive = np.zeros(1, dtype=np.float64)
        cart_comm.Allreduce(
            send, receive, op=MPI.SUM
        )
        return float(receive[0])

    # Initial global thermal energy (step 0).
    initial_host = u_device.copy_to_host()
    local_energy = (
        np.sum(initial_host[1:ny+1, 1:nx+1])
        * dx * dy
    )

    energy_trace = [global_sum(local_energy)]
    residual_trace = [0.0]

    cart_comm.Barrier()
    start_time = MPI.Wtime()

    for step in range(1, steps + 1):

        if mode == "serialized":
            step_serialized(
                u_device, u_new_device,
                kappa_device, source_device,
                context, dt, dx, dy,
                stream_boundary
            )

        elif mode == "overlapped":
            step_overlapped(
                u_device, u_new_device,
                kappa_device, source_device,
                context, dt, dx, dy,
                stream_boundary, stream_interior
            )

        else:
            raise ValueError("Unknown solver mode: " + mode)

        # Residual is the L2 norm of the latest update:
        # sqrt(global sum of (u_new - u_old)^2).
        if step % 100 == 0:
            old_host = u_device.copy_to_host()
            new_host = u_new_device.copy_to_host()

            owned_old = old_host[1:ny+1, 1:nx+1]
            owned_new = new_host[1:ny+1, 1:nx+1]

            local_residual_sq = float(
                np.sum((owned_new - owned_old) ** 2)
            )

            global_residual = np.sqrt(
                global_sum(local_residual_sq)
            )
            residual_trace.append(global_residual)

        # Swap current and next solution buffers.
        u_device, u_new_device = u_new_device, u_device

        if step % 100 == 0:
            current_host = u_device.copy_to_host()

            local_energy = (
                np.sum(current_host[1:ny+1, 1:nx+1])
                * dx * dy
            )

            energy_trace.append(global_sum(local_energy))

    stream_boundary.synchronize()
    stream_interior.synchronize()

    cart_comm.Barrier()
    elapsed = MPI.Wtime() - start_time

    # Report the slowest rank's elapsed time.
    local_time = np.array(elapsed, dtype=np.float64)
    maximum_time = np.zeros(1, dtype=np.float64)

    cart_comm.Allreduce(
        local_time, maximum_time, op=MPI.MAX
    )

    final_state = u_device.copy_to_host()

    return {
        "final_state": final_state,
        "energy_trace": energy_trace,
        "residual_trace": residual_trace,
        "elapsed_ms": float(maximum_time[0] * 1000.0)
    }


def run_solver(context, steps=400):
    """Run serialized and overlapped simulations and compare results."""
    cart_comm = context["cart_comm"]
    rank = context["rank"]

    # Both modes use identical deterministic initialization.
    fields_serial = initialize_subdomains(context)
    fields_overlap = initialize_subdomains(context)

    local_kmax = float(np.max(fields_serial["kappa"]))
    kmax_send = np.array(local_kmax, dtype=np.float64)
    kmax_receive = np.zeros(1, dtype=np.float64)

    cart_comm.Allreduce(
        kmax_send, kmax_receive, op=MPI.MAX
    )

    kappa_max = float(kmax_receive[0])

    dx = fields_serial["dx"]
    dy = fields_serial["dy"]

    # Assignment's 2D explicit diffusion stability limit.
    dt_max = min(dx, dy) ** 2 / (4.0 * kappa_max)
    dt = 0.9 * dt_max

    if rank == 0:
        print("\n========== TASK 5 CONFIGURATION ==========")
        print("MPI ranks:", cart_comm.Get_size())
        print("Grid per rank:", (LOCAL_NY, LOCAL_NX))
        print("Time steps:", steps)
        print("Global maximum diffusivity:", kappa_max)
        print("dx:", dx)
        print("dy:", dy)
        print("dt maximum:", dt_max)
        print("dt used:", dt)

    serialized = _run_one_mode(
        context, "serialized", steps, dt, fields_serial
    )

    overlapped = _run_one_mode(
        context, "overlapped", steps, dt, fields_overlap
    )

    # Compare owned cells only; halo values are communication buffers.
    serial_owned = serialized["final_state"][
        1:LOCAL_NY+1, 1:LOCAL_NX+1
    ]

    overlap_owned = overlapped["final_state"][
        1:LOCAL_NY+1, 1:LOCAL_NX+1
    ]

    local_difference = float(
        np.max(np.abs(serial_owned - overlap_owned))
    )

    send_diff = np.array(local_difference, dtype=np.float64)
    receive_diff = np.zeros(1, dtype=np.float64)

    cart_comm.Allreduce(
        send_diff, receive_diff, op=MPI.MAX
    )

    parity_max_diff = float(receive_diff[0])

    if rank == 0:
        print("\n========== ENERGY AND RESIDUAL TRACES ==========")
        for index, step in enumerate(range(0, steps + 1, 100)):
            print(
                f"Step {step:3d} | "
                f"Serialized energy={serialized['energy_trace'][index]:.16e} | "
                f"Overlapped energy={overlapped['energy_trace'][index]:.16e} | "
                f"Serialized residual={serialized['residual_trace'][index]:.16e} | "
                f"Overlapped residual={overlapped['residual_trace'][index]:.16e}"
            )

        print("\n========== TASK 5 BENCHMARK ==========")
        print(
            f"Serialized time: {serialized['elapsed_ms']:.3f} ms"
        )
        print(
            f"Overlapped time: {overlapped['elapsed_ms']:.3f} ms"
        )
        print(
            f"Parity max difference: {parity_max_diff:.16e}"
        )

        if parity_max_diff < 1e-12:
            print("TASK 5 NUMERICAL PARITY: PASS")
        else:
            print("TASK 5 NUMERICAL PARITY: FAIL")

    return {
        "dt": dt,
        "dt_max": dt_max,
        "kappa_max": kappa_max,
        "serialized": serialized,
        "overlapped": overlapped,
        "parity_max_diff": parity_max_diff
    }

'''

marker = 'if __name__ == "__main__":'

if marker not in source_text:
    raise RuntimeError("Could not find the main execution section.")

source_text = source_text.replace(
    marker,
    new_code + "\n\n" + marker,
    1
)

ast.parse(source_text)
path.write_text(source_text)

print("Task 5 solver functions inserted.")
print("Python syntax: VALID")
print("step_serialized: FOUND")
print("_run_one_mode: FOUND")
print("run_solver: FOUND")


Task 5 solver functions inserted.
Python syntax: VALID
step_serialized: FOUND
_run_one_mode: FOUND
run_solver: FOUND


In [47]:

%%writefile test_task5.py

import json
from distributed_cuda_solver import (
    init_distributed_context,
    run_solver
)

github_user = "comepotyyy"

context = init_distributed_context(github_user)

results = run_solver(context, steps=400)

if context["rank"] == 0:
    summary = {
        "mpi_ranks": context["cart_comm"].Get_size(),
        "steps": 400,
        "kappa_max": results["kappa_max"],
        "dt_max": results["dt_max"],
        "dt": results["dt"],
        "serialized_ms": results["serialized"]["elapsed_ms"],
        "overlapped_ms": results["overlapped"]["elapsed_ms"],
        "parity_max_diff": results["parity_max_diff"],
        "serialized_energy": results["serialized"]["energy_trace"],
        "overlapped_energy": results["overlapped"]["energy_trace"],
        "serialized_residual": results["serialized"]["residual_trace"],
        "overlapped_residual": results["overlapped"]["residual_trace"]
    }

    with open("task5_results.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nTask 5 results saved to task5_results.json")


Writing test_task5.py


In [48]:

import os
import sys
import subprocess

clean_env = os.environ.copy()

for key in list(clean_env.keys()):
    if key.startswith(("OMPI_", "PMIX_", "ORTE_", "OPAL_", "PRTE_")):
        clean_env.pop(key)

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--mca", "ess", "hnp",
    "--tag-output",
    "-np", "4",
    sys.executable,
    "test_task5.py"
]

print("Starting 400-step serialized and overlapped simulations...")

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=900,
    env=clean_env
)

print("Return code:", result.returncode)
print("\n========== STDOUT ==========")
print(result.stdout or "(empty)")
print("\n========== STDERR ==========")
print(result.stderr or "(empty)")


Starting 400-step serialized and overlapped simulations...
Return code: 0

========== STDOUT ==========
[1,1]<stdout>:[Rank 1] Coordinates=[0, 1], Up=-2, Down=3, Left=0, Right=-2, GPU=0, Seed=7843665258511339110
[1,0]<stdout>:[Rank 0] Coordinates=[0, 0], Up=-2, Down=2, Left=-2, Right=1, GPU=0, Seed=7843665258511339110
[1,3]<stdout>:[Rank 3] Coordinates=[1, 1], Up=1, Down=-2, Left=2, Right=-2, GPU=0, Seed=7843665258511339110
[1,2]<stdout>:[Rank 2] Coordinates=[1, 0], Up=0, Down=-2, Left=-2, Right=3, GPU=0, Seed=7843665258511339110
[1,0]<stdout>:
[1,0]<stdout>:========== TASK 5 CONFIGURATION ==========
[1,0]<stdout>:MPI ranks: 4
[1,0]<stdout>:Grid per rank: (256, 256)
[1,0]<stdout>:Time steps: 400
[1,0]<stdout>:Global maximum diffusivity: 0.2003008277993342
[1,0]<stdout>:dx: 0.001949317738791423
[1,0]<stdout>:dy: 0.001949317738791423
[1,0]<stdout>:dt maximum: 4.7426659296856095e-06
[1,0]<stdout>:dt used: 4.268399336717049e-06
[1,0]<stdout>:
[1,0]<stdout>:========== ENERGY AND RESIDUAL TR

In [49]:

import subprocess
import json
import os
import sys
import numpy as np
from numba import cuda
from mpi4py import MPI

print("========== HARDWARE TELEMETRY ==========")

gpu_info = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,uuid,compute_cap,memory.total,driver_version",
        "--format=csv,noheader"
    ],
    text=True
).strip()

print("GPU:", gpu_info)

print("\nCUDA device:")
device = cuda.get_current_device()
print("Device name:", device.name)
print("Compute capability:", device.compute_capability)

print("\nMPI:")
print("MPI library:", MPI.Get_library_version().strip())
print("mpi4py version:", __import__("mpi4py").__version__)

print("\nPython:", sys.version)
print("NumPy:", np.__version__)

print("\n========== TASK 5 RESULTS FILE ==========")

with open("task5_results.json", "r") as f:
    results = json.load(f)

print(json.dumps(results, indent=2))

print("\n========== FILE CHECK ==========")

for filename in [
    "distributed_cuda_solver.py",
    "test_task4_overlapped.py",
    "test_task5.py",
    "task5_results.json"
]:
    print(filename, "EXISTS" if os.path.exists(filename) else "MISSING")


========== HARDWARE TELEMETRY ==========
GPU: Tesla T4, GPU-9327fcef-38d8-daf7-d486-e79cefc0275a, 7.5, 15360 MiB, 580.82.07

CUDA device:
Device name: Tesla T4
Compute capability: ComputeCapability(major=7, minor=5)

MPI:
MPI library: Open MPI v4.1.6, package: Debian OpenMPI, ident: 4.1.6, repo rev: v4.1.6, Sep 30, 2023 
mpi4py version: 4.1.2

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 2.4.6

========== TASK 5 RESULTS FILE ==========
{
  "mpi_ranks": 4,
  "steps": 400,
  "kappa_max": 0.2003008277993342,
  "dt_max": 4.7426659296856095e-06,
  "dt": 4.268399336717049e-06,
  "serialized_ms": 8579.875382,
  "overlapped_ms": 5387.814107,
  "parity_max_diff": 0.0,
  "serialized_energy": [
    0.1002549323591333,
    0.10022387336422295,
    0.10020463726592138,
    0.10018711052273863,
    0.10017014691705371
  ],
  "overlapped_energy": [
    0.1002549323591333,
    0.10022387336422295,
    0.10020463726592138,
    0.10018711052273863,
    0.10017014691705371
  ],
  "se

In [50]:

import json
import subprocess
import sys
import numpy as np
import mpi4py
from numba import cuda

with open("task5_results.json", "r") as f:
    results = json.load(f)

gpu_info = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,uuid,compute_cap,memory.total,driver_version",
        "--format=csv,noheader"
    ],
    text=True
).strip()

serialized_ms = results["serialized_ms"]
overlapped_ms = results["overlapped_ms"]

manifest = {
    "github_username": "comepotyyy",
    "instance_seed": 7843665258511339110,

    "hardware": {
        "gpu": "NVIDIA Tesla T4",
        "gpu_uuid": "GPU-9327fcef-38d8-daf7-d486-e79cefc0275a",
        "compute_capability": "7.5",
        "memory": "15360 MiB",
        "driver": "580.82.07",
        "telemetry": gpu_info
    },

    "software": {
        "python": sys.version,
        "numpy": np.__version__,
        "mpi": "Open MPI 4.1.6",
        "mpi4py": mpi4py.__version__
    },

    "distributed_configuration": {
        "mpi_ranks": 4,
        "topology": "2x2 Cartesian",
        "periodic": False,
        "global_grid": [512, 512],
        "local_grid_per_rank": [256, 256],
        "gpu_device_per_rank": 0
    },

    "simulation": {
        "steps": 400,
        "kappa_max": results["kappa_max"],
        "dx": 1.0 / 513.0,
        "dy": 1.0 / 513.0,
        "dt_max": results["dt_max"],
        "dt_used": results["dt"]
    },

    "ptx_audit": {
        "st.shared.f64": 5,
        "ld.shared.f64": 5,
        "bar.sync": 1,
        "ptx_audit_passed": True
    },

    "benchmark": {
        "serialized_ms": serialized_ms,
        "overlapped_ms": overlapped_ms,
        "speedup": serialized_ms / overlapped_ms,
        "time_saved_percent": (
            (serialized_ms - overlapped_ms) / serialized_ms
        ) * 100
    },

    "numerical_verification": {
        "parity_max_diff": results["parity_max_diff"],
        "parity_tolerance": 1e-12,
        "parity_passed": results["parity_max_diff"] < 1e-12
    },

    "energy_trace_steps": [0, 100, 200, 300, 400],
    "serialized_energy": results["serialized_energy"],
    "overlapped_energy": results["overlapped_energy"],
    "serialized_residual": results["serialized_residual"],
    "overlapped_residual": results["overlapped_residual"],

    "task_status": {
        "task_1_topology": "PASS",
        "task_2_initialization": "PASS",
        "task_3_halo_exchange": "PASS",
        "task_4_tiled_stencil": "PASS",
        "task_5_benchmark_and_parity": "PASS"
    }
}

with open("submission_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("submission_manifest.json created successfully.")
print(json.dumps(manifest, indent=2))


submission_manifest.json created successfully.
{
  "github_username": "comepotyyy",
  "instance_seed": 7843665258511339110,
  "hardware": {
    "gpu": "NVIDIA Tesla T4",
    "gpu_uuid": "GPU-9327fcef-38d8-daf7-d486-e79cefc0275a",
    "compute_capability": "7.5",
    "memory": "15360 MiB",
    "driver": "580.82.07",
    "telemetry": "Tesla T4, GPU-9327fcef-38d8-daf7-d486-e79cefc0275a, 7.5, 15360 MiB, 580.82.07"
  },
  "software": {
    "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
    "numpy": "2.4.6",
    "mpi": "Open MPI 4.1.6",
    "mpi4py": "4.1.2"
  },
  "distributed_configuration": {
    "mpi_ranks": 4,
    "topology": "2x2 Cartesian",
    "periodic": false,
    "global_grid": [
      512,
      512
    ],
    "local_grid_per_rank": [
      256,
      256
    ],
    "gpu_device_per_rank": 0
  },
  "simulation": {
    "steps": 400,
    "kappa_max": 0.2003008277993342,
    "dx": 0.001949317738791423,
    "dy": 0.001949317738791423,
    "dt_max": 4.7426659296856095e

In [52]:

import os
import json

required_files = [
    "mpi_cuda_stencil_lab.ipynb",
    "distributed_cuda_solver.py",
    "submission_manifest.json"
]

print("========== SUBMISSION FILE CHECK ==========")

for filename in required_files:
    exists = os.path.isfile(filename)
    size = os.path.getsize(filename) if exists else 0
    print(
        f"{filename}: "
        f"{'FOUND' if exists else 'MISSING'}"
        + (f" | {size:,} bytes" if exists else "")
    )

with open("submission_manifest.json", "r") as f:
    manifest = json.load(f)

print("\n========== REQUIRED VERIFICATION ==========")
print("MPI ranks:", manifest["distributed_configuration"]["mpi_ranks"])
print("PTX audit:", manifest["ptx_audit"]["ptx_audit_passed"])
print("Parity:", manifest["numerical_verification"]["parity_max_diff"])
print("All tasks:", manifest["task_status"])


========== SUBMISSION FILE CHECK ==========
mpi_cuda_stencil_lab.ipynb: MISSING
distributed_cuda_solver.py: FOUND | 24,416 bytes
submission_manifest.json: FOUND | 2,620 bytes

========== REQUIRED VERIFICATION ==========
MPI ranks: 4
PTX audit: True
Parity: 0.0
All tasks: {'task_1_topology': 'PASS', 'task_2_initialization': 'PASS', 'task_3_halo_exchange': 'PASS', 'task_4_tiled_stencil': 'PASS', 'task_5_benchmark_and_parity': 'PASS'}
